In [11]:
from pathlib import Path

import pandas as pd
import plotly.express as px

# Permite ejecutar el notebook desde la raíz o desde notebooks/.
RAIZ = Path.cwd()
if not (RAIZ / "datos").is_dir():
    RAIZ = RAIZ.parent

RUTA_DATOS = RAIZ / "datos" / "procesados"

paro = pd.read_csv(
    RUTA_DATOS / "paro_total_mensual.csv",
    sep=";",
    encoding="utf-8-sig",
    parse_dates=["fecha"],
)

edades = pd.read_csv(
    RUTA_DATOS / "paro_edad_mensual.csv",
    sep=";",
    encoding="utf-8-sig",
    parse_dates=["fecha"],
)

paro = paro.sort_values(["municipio", "fecha"]).reset_index(drop=True)
edades = edades.sort_values(
    ["municipio", "grupo_edad", "fecha"]
).reset_index(drop=True)

print("Filas de paro total:", len(paro))
print("Municipios:", paro["municipio"].nunique())
print("Primera fecha:", paro["fecha"].min())
print("Última fecha:", paro["fecha"].max())

display(paro.head())

Filas de paro total: 46898
Municipios: 179
Primera fecha: 2005-01-01 00:00:00
Última fecha: 2026-10-01 00:00:00


,municipio,fecha,paro_total
0,"Acebeda, La",2005-01-01,3
1,"Acebeda, La",2005-02-01,2
2,"Acebeda, La",2005-03-01,2
3,"Acebeda, La",2005-04-01,2
4,"Acebeda, La",2005-05-01,2


In [12]:
assert not paro.duplicated(["municipio", "fecha"]).any()

for municipio, grupo in paro.groupby("municipio"):
    meses = pd.date_range(
        paro["fecha"].min(),
        paro["fecha"].max(),
        freq="MS",
    )
    assert pd.DatetimeIndex(grupo["fecha"]).equals(meses), municipio

print("Cada municipio tiene una observación por mes, sin huecos.")

Cada municipio tiene una observación por mes, sin huecos.


In [13]:
series = paro.groupby("municipio")["paro_total"]

paro["paro_mes_anterior"] = series.shift(1)
paro["paro_mismo_mes_anterior_ano"] = series.shift(12)

paro["cambio_mensual"] = (
    paro["paro_total"] - paro["paro_mes_anterior"]
)

paro["cambio_interanual"] = (
    paro["paro_total"] - paro["paro_mismo_mes_anterior_ano"]
)

# No calculamos porcentajes cuando el denominador es cero.
base_mensual = paro["paro_mes_anterior"].where(
    paro["paro_mes_anterior"] > 0
)
base_anual = paro["paro_mismo_mes_anterior_ano"].where(
    paro["paro_mismo_mes_anterior_ano"] > 0
)

paro["variacion_mensual_pct"] = (
    100 * paro["cambio_mensual"] / base_mensual
)

paro["variacion_interanual_pct"] = (
    100 * paro["cambio_interanual"] / base_anual
)

display(paro.head(15))

,municipio,fecha,paro_total,paro_mes_anterior,paro_mismo_mes_anterior_ano,cambio_mensual,cambio_interanual,variacion_mensual_pct,variacion_interanual_pct
0,"Acebeda, La",2005-01-01,3,NaN,NaN,NaN,NaN,NaN,NaN
1,"Acebeda, La",2005-02-01,2,3.0,NaN,-1.0,NaN,-33.333333,NaN
2,"Acebeda, La",2005-03-01,2,2.0,NaN,0.0,NaN,0.000000,NaN
3,"Acebeda, La",2005-04-01,2,2.0,NaN,0.0,NaN,0.000000,NaN
4,"Acebeda, La",2005-05-01,2,2.0,NaN,0.0,NaN,0.000000,NaN
5,"Acebeda, La",2005-06-01,2,2.0,NaN,0.0,NaN,0.000000,NaN
6,"Acebeda, La",2005-07-01,0,2.0,NaN,-2.0,NaN,-100.000000,NaN
7,"Acebeda, La",2005-08-01,1,0.0,NaN,1.0,NaN,NaN,NaN
8,"Acebeda, La",2005-09-01,0,1.0,NaN,-1.0,NaN,-100.000000,NaN
9,"Acebeda, La",2005-10-01,0,0.0,NaN,0.0,NaN,NaN,NaN


In [14]:
MUNICIPIOS = [
    "Madrid",
    "Alcalá de Henares",
    "Aranjuez",
    "Acebeda, La",
]

muestra = paro[paro["municipio"].isin(MUNICIPIOS)].copy()

fig = px.line(
    muestra,
    x="fecha",
    y="paro_total",
    facet_row="municipio",
    title="Evolución del paro registrado por municipio",
    labels={"fecha": "Mes", "paro_total": "Personas paradas"},
    height=900,
)

fig.update_yaxes(matches=None)
fig.show()

### Observaciones sobre la evolución del paro registrado

1. **Patrón temporal compartido.** Madrid, Alcalá de Henares y
   Aranjuez presentan una evolución similar: un aumento acusado
   a partir de aproximadamente 2008, niveles máximos alrededor
   de 2013 y un descenso posterior hasta 2019. En torno a
   2020–2021 se observa un nuevo repunte, seguido de otra
   fase descendente.

2. **La evolución no es uniforme.** Los descensos de largo plazo
   contienen subidas y bajadas mensuales. Por tanto, una subida
   aislada no implica necesariamente un cambio de tendencia
   ni un comportamiento anómalo.

3. **Diferencias de escala.** Los municipios presentan niveles
   de paro registrado muy distintos. Además, los gráficos
   utilizan escalas verticales independientes: permiten
   comparar la forma de las series, pero no directamente
   la magnitud visual de sus oscilaciones. Estos recuentos
   no permiten comparar tasas de paro municipales.

4. **Particularidades de La Acebeda.** La serie presenta
   recuentos muy pequeños, incluidos meses con cero personas
   paradas. Los cambios de unas pocas personas pueden generar
   variaciones porcentuales elevadas. Será necesario considerar
   conjuntamente el cambio absoluto y el relativo, evitando
   calcular porcentajes cuando el valor de referencia sea cero.

5. **Estacionalidad pendiente de comprobar.** Se observan
   oscilaciones de corto plazo, pero este gráfico general
   no permite confirmar si se repiten sistemáticamente
   en los mismos meses. Se estudiarán mediante comparaciones
   entre meses y años.

6. **Implicaciones para las alertas.** La identificación de
   aumentos inusuales deberá considerar el comportamiento
   histórico de cada municipio, su tendencia, su posible
   estacionalidad y la variabilidad de sus cambios.
   Un mismo umbral absoluto o porcentual podría comportarse
   de forma muy diferente entre municipios.

Estas observaciones son descriptivas y se refieren a los cuatro
municipios seleccionados. No demuestran las causas de los cambios
ni permiten generalizar los resultados a todos los municipios.

In [15]:
MUNICIPIO = "Madrid"

detalle = paro[
    paro["municipio"].eq(MUNICIPIO)
    & paro["fecha"].dt.year.between(2015, 2019)
].copy()

detalle["mes"] = detalle["fecha"].dt.month
detalle["ano"] = detalle["fecha"].dt.year.astype(str)

fig = px.line(
    detalle,
    x="mes",
    y="variacion_mensual_pct",
    color="ano",
    markers=True,
    title=f"{MUNICIPIO}: variación mensual por mes del año",
    labels={
        "mes": "Mes",
        "variacion_mensual_pct": "Variación mensual (%)",
        "ano": "Año",
    },
)

fig.update_xaxes(dtick=1)
fig.add_hline(y=0, line_dash="dash")
fig.show()

### Exploración de la estacionalidad: Madrid, 2015–2019

Las variaciones mensuales presentan un patrón similar en los cinco
años analizados:

- Enero y febrero registran aumentos del paro, especialmente enero.
- Entre marzo y junio predominan los descensos, con las caídas
  más pronunciadas en abril o mayo.
- Agosto presenta un aumento en todos los años observados.
- Noviembre y diciembre registran descensos en los cinco años.
- Julio y octubre muestran diferencias entre años, tanto en la
  magnitud como en el signo de la variación.

La repetición de estas pautas constituye evidencia visual compatible
con estacionalidad. Sin embargo, este análisis no separa formalmente
el componente estacional de la tendencia y de otros cambios.

### Implicaciones para las alertas

Una subida del paro en enero o agosto puede formar parte del patrón
habitual de Madrid. Por tanto, no debería clasificarse como inusual
únicamente por ser positiva: será necesario comparar su magnitud
con la esperada para ese municipio y mes.

Del mismo modo, un descenso menor de lo habitual podría indicar
un comportamiento peor de lo esperado, aunque el paro no aumente
respecto al mes anterior. Debemos distinguir este caso de una
alerta por aumento efectivo del paro.

Estas conclusiones se limitan a Madrid y al periodo 2015–2019.
Queda pendiente comprobar si el patrón se mantiene en años
posteriores y si aparece en otros municipios.

In [16]:
MUNICIPIO = "Madrid"

detalle = paro[
    paro["municipio"].eq(MUNICIPIO)
    & paro["fecha"].dt.year.between(2021, 2025)
].copy()

detalle["mes"] = detalle["fecha"].dt.month
detalle["ano"] = detalle["fecha"].dt.year.astype(str)

fig = px.line(
    detalle,
    x="mes",
    y="variacion_mensual_pct",
    color="ano",
    markers=True,
    title=f"{MUNICIPIO}: variación mensual por mes del año",
    labels={
        "mes": "Mes",
        "variacion_mensual_pct": "Variación mensual (%)",
        "ano": "Año",
    },
)

fig.update_xaxes(dtick=1)
fig.add_hline(y=0, line_dash="dash")
fig.show()

### Comparación de los patrones mensuales: Madrid, 2021–2025

En este periodo se observan diferencias entre años, junto con algunos
comportamientos que se repiten:

- **Entre abril y junio, el paro registrado disminuye en todos los años
  representados.** También se observan descensos en noviembre y diciembre.
  Estos resultados mantienen parte del patrón observado en 2015–2019.

- **Agosto presenta aumentos en los cinco años**, aunque con distinta
  intensidad. Por tanto, una subida en este mes no debería considerarse
  inusual únicamente por ser positiva.

- **Enero y febrero muestran diferencias importantes.** En 2022 se
  registran descensos, especialmente pronunciados en febrero, mientras
  que en los demás años aparecen aumentos. El comportamiento de estos
  meses no es uniforme durante todo el periodo.

- **Destacan algunos descensos por su magnitud**, como diciembre de 2021
  y febrero de 2022. Son observaciones que conviene revisar en los datos
  antes de interpretarlas. El gráfico, por sí solo, no permite determinar
  sus causas ni clasificarlas formalmente como anomalías.

### Implicaciones para la detección de cambios inusuales

La comparación con 2015–2019 sugiere que existen patrones mensuales
recurrentes, pero también que su intensidad y, en algunos meses, su
dirección varían entre años.

Estas curvas muestran variaciones mensuales observadas; no aíslan el
componente estacional de la tendencia ni de otros movimientos
excepcionales. Por ello, no podemos concluir únicamente con estos
gráficos que haya cambiado la estacionalidad.

Para definir una alerta, será necesario comparar el cambio observado
con el comportamiento histórico del municipio en ese mes y considerar
su evolución reciente. Una subida habitual en agosto puede no ser
inusual, mientras que una subida en un mes que suele registrar descensos
puede merecer una revisión.

El método deberá utilizar exclusivamente información anterior al mes
evaluado y comprobar cómo afectan los años con movimientos excepcionales
a los resultados. También se tendrá en cuenta el cambio en número de
personas, además del porcentaje.

Estas conclusiones son exploratorias y corresponden al municipio de
Madrid; no deben generalizarse automáticamente al resto de municipios.

In [17]:
ultima_fecha = paro["fecha"].max()

ultimo_mes = paro[paro["fecha"].eq(ultima_fecha)].copy()

columnas = [
    "municipio",
    "paro_total",
    "paro_mes_anterior",
    "cambio_mensual",
    "variacion_mensual_pct",
    "cambio_interanual",
    "variacion_interanual_pct",
]

display(
    ultimo_mes.sort_values("cambio_mensual", ascending=False)
    [columnas]
    .head(10)
    .round(2)
)

,municipio,paro_total,paro_mes_anterior,cambio_mensual,variacion_mensual_pct,cambio_interanual,variacion_interanual_pct
20435,Madrid,138652,136522.0,2130.0,1.56,4334.0,3.23
23579,Móstoles,9445,9195.0,250.0,2.72,109.0,1.17
19125,Leganés,8455,8210.0,245.0,2.98,15.0,0.18
14933,Fuenlabrada,9208,8995.0,213.0,2.37,-59.0,-0.64
26461,Parla,7053,6864.0,189.0,2.75,74.0,1.06
3405,Arganda del Rey,3242,3104.0,138.0,4.45,30.0,0.93
16767,Getafe,8577,8456.0,121.0,1.43,356.0,4.33
1571,Alcorcón,7202,7107.0,95.0,1.34,174.0,2.48
11527,Collado Villalba,3273,3207.0,66.0,2.06,90.0,2.83
1309,Alcobendas,3906,3841.0,65.0,1.69,193.0,5.20


In [18]:
display(
    ultimo_mes.sort_values("variacion_mensual_pct", ascending=False)
    [columnas]
    .head(10)
    .round(2)
)

,municipio,paro_total,paro_mes_anterior,cambio_mensual,variacion_mensual_pct,cambio_interanual,variacion_interanual_pct
35107,"Serna del Monte, La",4,1.0,3.0,300.00,2.0,100.00
785,Alameda del Valle,20,11.0,9.0,81.82,0.0,0.00
35893,Somosierra,3,2.0,1.0,50.00,2.0,200.00
2881,Anchuelo,50,37.0,13.0,35.14,-8.0,-13.79
16505,Gascones,14,12.0,2.0,16.67,5.0,55.56
44015,Villamanrique de Tajo,66,57.0,9.0,15.79,16.0,32.00
42181,Valverde de Alcalá,31,27.0,4.0,14.81,6.0,24.00
34321,Santa María de la Alameda,117,102.0,15.0,14.71,5.0,4.46
46373,Villavieja de Lozoya,16,14.0,2.0,14.29,3.0,23.08
34583,Santorcaz,45,40.0,5.0,12.50,5.0,12.50


### Comparación entre aumentos absolutos y porcentuales

Las clasificaciones del último mes muestran resultados distintos según
se ordenen los municipios por el aumento en número de personas o por
la variación porcentual del paro registrado.

- **En términos absolutos**, Madrid presenta el mayor aumento:
  pasa de 136.522 a 138.652 personas paradas, un incremento de
  2.130 personas que equivale al 1,56 %. Le siguen Móstoles
  (+250 personas) y Leganés (+245 personas).

- **En términos porcentuales**, destacan municipios que parten de
  cifras reducidas. La Serna del Monte pasa de 1 a 4 personas paradas:
  un aumento del 300 %, pero de solo 3 personas en términos absolutos.
  Somosierra pasa de 2 a 3 personas, lo que representa un 50 %.

- Alameda del Valle muestra otro ejemplo: pasa de 11 a 20 personas,
  un aumento de 9 personas equivalente al 81,82 %.

Estos resultados ilustran el efecto del denominador: cuando el número
de personas paradas del mes anterior es pequeño, un cambio de pocas
personas puede producir una variación porcentual muy elevada.

### Implicaciones para las alertas

No debemos utilizar ninguna de estas clasificaciones por sí sola para
identificar cambios inusuales:

- El cambio absoluto permite conocer la magnitud del aumento en número
  de personas, pero tiende a destacar municipios con mayor volumen de paro.
- El porcentaje expresa el cambio respecto al nivel anterior, pero es
  muy sensible a las bases pequeñas.
- Ninguna de las dos medidas determina si el cambio se aparta del
  comportamiento habitual del municipio.

Por tanto, las alertas deberán considerar conjuntamente el cambio
absoluto, el porcentaje, el nivel de partida y el comportamiento
histórico y estacional de cada municipio. Los casos con bases pequeñas
deberán señalarse para facilitar su interpretación, sin descartarlos
automáticamente.

También conviene distinguir la evolución mensual de la interanual.
Por ejemplo, Anchuelo aumenta en 13 personas respecto al mes anterior,
pero presenta 8 personas menos que en el mismo mes del año anterior.
Ambas comparaciones aportan información complementaria.

Estas tablas son descriptivas y no constituyen todavía un listado
de anomalías. Además, la variación porcentual del paro registrado
no es una tasa de paro.

In [19]:
MUNICIPIO = "Madrid"

edades["cambio_mensual"] = (
    edades.groupby(["municipio", "grupo_edad"])["paro"].diff()
)

desglose = edades[
    edades["municipio"].eq(MUNICIPIO)
    & edades["fecha"].eq(ultima_fecha)
]

display(
    desglose[["municipio", "fecha", "grupo_edad", "paro", "cambio_mensual"]]
)

print(
    "Cambio total obtenido sumando los grupos:",
    desglose["cambio_mensual"].sum(min_count=3),
)

,municipio,fecha,grupo_edad,paro,cambio_mensual
60783,Madrid,2026-10-01,De 25 a 44 años,52955,1074.0
61045,Madrid,2026-10-01,Mayor de 44 años,76016,352.0
61307,Madrid,2026-10-01,Menor de 25 años,9681,704.0


Cambio total obtenido sumando los grupos: 2130.0


### Definición provisional de «cambio inusual»

Consideraremos un cambio inusual una variación mensual del paro
registrado que se aleje del comportamiento esperado para ese municipio,
teniendo en cuenta el mes del año, su evolución reciente y la variabilidad
observada en su historial.

El proyecto se centrará inicialmente en detectar aumentos inusuales:
meses en los que el paro registrado aumenta y ese aumento supera
lo esperable según la información histórica disponible.

### Criterios que orientarán la detección

1. **Comparación con el propio municipio.**
   Cada municipio se evaluará respecto a su historial. Un mismo aumento
   puede ser habitual en un municipio y excepcional en otro.

2. **Estacionalidad y evolución reciente.**
   Una subida no constituye por sí sola una anomalía. Los gráficos
   muestran aumentos recurrentes en determinados meses, pero también
   diferencias entre periodos. La referencia deberá considerar ambos
   aspectos.

3. **Variabilidad habitual.**
   La distancia entre el cambio observado y el esperado se interpretará
   teniendo en cuenta cuánto suelen fluctuar los datos del municipio.

4. **Distinción entre excepcionalidad y magnitud.**
   Se mostrará tanto lo inusual del cambio como su tamaño en número
   de personas y en porcentaje. Un aumento grande no es necesariamente
   excepcional, y uno excepcional no implica necesariamente un gran
   número de personas.

5. **Tratamiento de bases pequeñas y ceros.**
   Los porcentajes calculados sobre cifras reducidas requieren especial
   atención. Si el paro del mes anterior es cero, la variación porcentual
   no estará definida y no se sustituirá por cero. Estos casos deberán
   conservarse y evaluarse mediante un tratamiento específico.

6. **Desglose por edad.**
   Los grupos de edad permitirán describir qué parte del cambio total
   corresponde a cada grupo. Su contribución al aumento no demuestra,
   por sí sola, que su evolución sea inusual.

### Alcance de las alertas

Una alerta señalará un caso que merece revisión; no demostrará una
causa económica concreta ni un error en los datos.

También puede ocurrir que el paro disminuya menos de lo esperado.
Ese caso supone una evolución peor que la referencia, pero se
distinguirá de una alerta por aumento efectivo del paro.

### Aspectos pendientes de validación

Esta definición es conceptual y provisional. Falta elegir el método
para estimar el comportamiento esperado, la cantidad de historial
utilizada y el umbral para considerar excepcional una desviación.

Estas decisiones se evaluarán en meses históricos, utilizando para
cada mes únicamente información anterior a él. Se comprobará la
sensibilidad de los resultados a los parámetros y el comportamiento
del método en municipios con distintos niveles de paro.